# Experimento 03 — Remoção do `cibil_score`

**Dataset:** [Loan Approval Prediction Dataset](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset) · **Algoritmo:** `DecisionTreeClassifier`

**Pergunta:** O quanto o modelo depende do score de crédito? Sem ele, as outras variáveis conseguem prever a aprovação?

| Mudança | Aplicada |
|---|:-:|
| Tratar valores inválidos (remover linhas) | ✅ |
| Remover variáveis fracas | — |
| Remover `cibil_score` | ✅ |
| Undersampling no treino | — |

O cenário é comparado com dois modelos de referência treinados neste mesmo notebook: **Original** (como no notebook base) e **Dados limpos** (só o tratamento de valores inválidos). Todos usam a mesma divisão treino/teste e a mesma grade de hiperparâmetros.

## Importação das bibliotecas

In [ ]:
# !pip install kagglehub scikit-learn imbalanced-learn pandas matplotlib seaborn  # descomente se necessário

import glob
import os

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from matplotlib.patches import Patch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Configuração do cenário

Altere estes valores para testar outras variações.

In [ ]:
NOME_CENARIO = "03 · Sem cibil_score"
REMOVER_COLUNAS = ["cibil_score"]
UNDERSAMPLING = False

## Carregamento e codificação dos dados

In [ ]:
path = kagglehub.dataset_download("architsharma01/loan-approval-prediction-dataset")
csv_path = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_path)

# Os nomes das colunas e os valores de texto vêm com espaços no início
df.columns = df.columns.str.strip()
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

data = df.drop(columns=["loan_id"]).copy()
data["education"] = data["education"].map({"Graduate": 1, "Not Graduate": 0})
data["self_employed"] = data["self_employed"].map({"Yes": 1, "No": 0})
data["loan_status"] = data["loan_status"].map({"Approved": 1, "Rejected": 0})
assert data.isnull().sum().sum() == 0, "Há valores não mapeados!"

print("Formato:", data.shape)
data["loan_status"].value_counts().rename(index={1: "Approved", 0: "Rejected"})

## Tratamento de valores inválidos

Verificamos valores que não fazem sentido no domínio do problema:

- **patrimônio negativo** em qualquer uma das 4 colunas de ativos;
- `cibil_score` fora da faixa válida do score (300 a 900);
- prazo, renda ou valor do empréstimo menores ou iguais a zero;
- linhas duplicadas (todas as variáveis iguais).

As linhas com qualquer um desses problemas são **removidas**.

In [ ]:
COLS_ATIVOS = ["residential_assets_value", "commercial_assets_value",
               "luxury_assets_value", "bank_asset_value"]

checagens = {
    "patrimônio negativo": (data[COLS_ATIVOS] < 0).any(axis=1),
    "cibil_score fora de 300–900": ~data["cibil_score"].between(300, 900),
    "loan_term <= 0": data["loan_term"] <= 0,
    "income_annum <= 0": data["income_annum"] <= 0,
    "loan_amount <= 0": data["loan_amount"] <= 0,
    "linha duplicada": data.duplicated(),
}
invalido = pd.concat(checagens, axis=1).any(axis=1)

print(pd.Series({k: int(v.sum()) for k, v in checagens.items()}, name="linhas").to_string())
print(f"\nTotal de linhas inválidas: {invalido.sum()} ({invalido.mean():.2%} do dataset)")
print("\nValores negativos por coluna de ativos:")
print((data[COLS_ATIVOS] < 0).sum().to_string())

In [ ]:
# A remoção afeta as duas classes de forma parecida?
pd.DataFrame({
    "todas as linhas": data["loan_status"].value_counts(normalize=True),
    "linhas inválidas": data.loc[invalido, "loan_status"].value_counts(normalize=True),
}).rename(index={1: "Approved", 0: "Rejected"}).round(3)

## Divisão treino/teste

Para que **todos os cenários sejam comparados no mesmo conjunto de teste**:

1. a divisão 80/20 (estratificada, `random_state=42`) é feita **uma única vez**, sobre o dataset completo;
2. as linhas inválidas são retiradas do treino (nos cenários com tratamento) e **sempre retiradas do teste**.

Assim, até o modelo *Original* (treinado com as linhas inválidas) é avaliado nas mesmas linhas de teste que os demais, e as diferenças nas métricas vêm apenas das mudanças feitas no treino.

In [ ]:
X_todas = data.drop(columns=["loan_status"])
y_todas = data["loan_status"]

idx_treino, idx_teste = train_test_split(
    data.index, test_size=0.2, stratify=y_todas, random_state=RANDOM_STATE
)
validos = data.index[~invalido]
idx_treino_limpo = idx_treino.intersection(validos)
idx_teste_limpo = idx_teste.intersection(validos)

print("Treino (com inválidas):", len(idx_treino))
print("Treino (limpo):        ", len(idx_treino_limpo))
print("Teste (limpo):         ", len(idx_teste_limpo))

## Funções de treino e avaliação

`treinar_cenario` executa o mesmo `GridSearchCV` do notebook original (validação cruzada estratificada de 5 folds, otimizando F1) e avalia o melhor modelo no conjunto de teste.

Com `undersampling=True`, um `RandomUnderSampler` entra **dentro do pipeline**: em cada fold da validação cruzada, só a parte de treino é balanceada, e o **teste mantém a proporção real das classes**. Nesse caso, `class_weight` sai da grade, já que com as classes balanceadas ele não tem efeito.

Além das métricas do notebook original, medimos o **recall de cada classe** separadamente: `recall_rejeitado` é a fração dos pedidos que deveriam ser rejeitados e foram de fato rejeitados.

In [ ]:
PARAM_GRID = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "class_weight": [None, "balanced"],
}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
METRICAS_TESTE = ["acurácia_teste", "precisão", "recall_aprovado", "recall_rejeitado", "f1", "roc_auc"]


def treinar_cenario(nome, linhas_treino, remover=(), undersampling=False):
    colunas = [c for c in X_todas.columns if c not in remover]
    X_tr, y_tr = X_todas.loc[linhas_treino, colunas], y_todas.loc[linhas_treino]
    X_te, y_te = X_todas.loc[idx_teste_limpo, colunas], y_todas.loc[idx_teste_limpo]

    etapas = [("arvore", DecisionTreeClassifier(random_state=RANDOM_STATE))]
    grade = {f"arvore__{k}": v for k, v in PARAM_GRID.items()}
    if undersampling:
        etapas.insert(0, ("under", RandomUnderSampler(random_state=RANDOM_STATE)))
        del grade["arvore__class_weight"]

    busca = GridSearchCV(Pipeline(etapas), grade, scoring="f1", cv=CV, n_jobs=-1)
    busca.fit(X_tr, y_tr)
    modelo = busca.best_estimator_
    arvore = modelo.named_steps["arvore"]

    y_pred = modelo.predict(X_te)
    metricas = pd.Series({
        "linhas_treino": 2 * y_tr.value_counts().min() if undersampling else len(y_tr),
        "nº_variáveis": len(colunas),
        "acurácia_treino": accuracy_score(y_tr, modelo.predict(X_tr)),
        "acurácia_teste": accuracy_score(y_te, y_pred),
        "precisão": precision_score(y_te, y_pred),
        "recall_aprovado": recall_score(y_te, y_pred),
        "recall_rejeitado": recall_score(y_te, y_pred, pos_label=0),
        "f1": f1_score(y_te, y_pred),
        "roc_auc": roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1]),
        "f1_cv_média": busca.best_score_,
        "f1_cv_desvio": busca.cv_results_["std_test_score"][busca.best_index_],
        "profundidade": arvore.get_depth(),
        "nº_folhas": arvore.get_n_leaves(),
    }, name=nome)
    params = {k.removeprefix("arvore__"): v for k, v in busca.best_params_.items()}

    return {"nome": nome, "modelo": modelo, "arvore": arvore, "colunas": colunas,
            "X_teste": X_te, "y_teste": y_te, "params": params, "metricas": metricas}


def tabela_metricas(resultados):
    return pd.concat([r["metricas"] for r in resultados], axis=1).T


def tabela_params(resultados):
    return pd.DataFrame({r["nome"]: r["params"] for r in resultados}).astype(str)


def grafico_metricas(resultados):
    tabela = tabela_metricas(resultados)[METRICAS_TESTE]
    n = len(tabela)
    deslocamentos = np.linspace(-0.2, 0.2, n) if n > 1 else [0.0]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, (nome, linha) in enumerate(tabela.iterrows()):
        ax.scatter(linha.values, np.arange(len(METRICAS_TESTE)) + deslocamentos[i],
                   s=80, color=CORES[i], label=nome, zorder=3)
    ax.set_yticks(range(len(METRICAS_TESTE)), METRICAS_TESTE)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Valor no conjunto de teste")
    ax.set_title("Métricas por cenário")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    plt.tight_layout()
    plt.show()


def matrizes_confusao(resultados, colunas_grade=4):
    n = len(resultados)
    ncols = min(colunas_grade, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows), squeeze=False)
    for ax, r in zip(axes.ravel(), resultados):
        ConfusionMatrixDisplay.from_estimator(
            r["modelo"], r["X_teste"], r["y_teste"],
            display_labels=["Rejected", "Approved"], cmap="Blues", colorbar=False, ax=ax,
        )
        ax.set_title(r["nome"], fontsize=10)
        ax.grid(False)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def tabela_importancias(resultados):
    return pd.concat(
        {r["nome"]: pd.Series(r["arvore"].feature_importances_, index=r["colunas"]) for r in resultados},
        axis=1,
    ).reindex(X_todas.columns)

## Modelos de referência

- **Original:** mesmas condições do notebook `arvore_decisao_aprovacao_emprestimo.ipynb` (dados sem tratamento, todas as variáveis).
- **Dados limpos:** linhas inválidas removidas, todas as variáveis, sem balanceamento.

In [ ]:
original = treinar_cenario("Original", idx_treino)
limpo = treinar_cenario("Dados limpos", idx_treino_limpo)
tabela_metricas([original, limpo]).round(4)

## Preparação específica do cenário

### Variáveis removidas

Em laranja, as variáveis que este cenário retira do modelo, sobre as importâncias do modelo *Dados limpos*.

In [ ]:
# Importância das variáveis no modelo de referência (Dados limpos)
imp = pd.Series(limpo["arvore"].feature_importances_, index=limpo["colunas"]).sort_values()
cores = [CORES[1] if c in REMOVER_COLUNAS else CORES[0] for c in imp.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp.index, imp.values, color=cores)
ax.set_xlabel("Importância")
ax.set_title("Importância das variáveis no modelo 'Dados limpos'")
ax.legend(handles=[Patch(color=CORES[0], label="mantida"), Patch(color=CORES[1], label="removida neste cenário")],
          loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

imp.sort_values(ascending=False).round(4)

## Treino do cenário: 03 · Sem cibil_score

In [ ]:
cenario = treinar_cenario(NOME_CENARIO, idx_treino_limpo, remover=REMOVER_COLUNAS, undersampling=UNDERSAMPLING)
cenario["metricas"].round(4)

## Comparação dos cenários

In [ ]:
resultados = [original, limpo, cenario]
tabela_metricas(resultados).round(4)

In [ ]:
# Diferença do cenário em relação aos modelos de referência (positivo = o cenário é melhor)
metricas = tabela_metricas(resultados)
pd.DataFrame({
    "cenário − Original": metricas.loc[cenario["nome"]] - metricas.loc["Original"],
    "cenário − Dados limpos": metricas.loc[cenario["nome"]] - metricas.loc["Dados limpos"],
}).round(4)

### Métricas no conjunto de teste

In [ ]:
grafico_metricas(resultados)

### Matrizes de confusão

In [ ]:
matrizes_confusao(resultados)

### Hiperparâmetros escolhidos pelo GridSearchCV

In [ ]:
tabela_params(resultados)

### Importância das variáveis

Variáveis removidas no cenário aparecem como `NaN`.

In [ ]:
tabela_importancias(resultados).round(4)

### Regras da árvore — 03 · Sem cibil_score (até 3 níveis)

In [ ]:
print(export_text(cenario["arvore"], feature_names=cenario["colunas"], max_depth=3))

## O que observar

**Como ler a comparação:** uma diferença no teste só é relevante se for maior que a variação natural do modelo. Use a coluna `f1_cv_desvio` como referência: se a diferença de F1 entre dois cenários for menor que esse desvio, ela pode ser apenas efeito da amostra. O teste tem cerca de 850 linhas, então 0,01 de acurácia corresponde a cerca de 8 pedidos.

- A expectativa é de uma **queda forte** em todas as métricas, já que o `cibil_score` domina as divisões da árvore.
- Veja na tabela de importâncias quais variáveis assumem o topo da árvore sem o score.
- Um `roc_auc` próximo de 0,5 indicaria que as demais variáveis quase não têm informação sobre a aprovação.
- O resultado mostra o **risco** de um modelo depender de uma única variável: se o score não estiver disponível ou tiver erro, o modelo deixa de funcionar.